#  Celda 0: Preparación del entorno

In [6]:
# Recargar automáticamente los módulos durante el desarrollo
%reload_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Definir las rutas del proyecto
SRC_DIR = Path("../src").resolve()
DATA_DIR = Path("../data")
TABLES_DIR = Path("../results/tables")
FIGURES_DIR = Path("../results/figures")

# Comprobar la existencia de la carpeta de scripts
if not SRC_DIR.is_dir():
    raise FileNotFoundError(
        "No se encontró la carpeta src del proyecto."
    )

# Permitir importar los módulos de src
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Crear las carpetas de salida
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Entorno preparado correctamente.")

Entorno preparado correctamente.


# Celda 1: Cargar y selección de DEGs

In [7]:
# Cargar los resultados completos de InMoose
expression_file = TABLES_DIR / "deg_results_completa.csv"

expression_df = pd.read_csv(expression_file)

# Seleccionar genes diferencialmente expresados
deg_mask = (
    (expression_df["adj_pvalue"] < 0.05)
    & (expression_df["log2FoldChange"].abs() > 1)
)

degs_df = expression_df.loc[deg_mask].copy()

# Mostrar un resumen
print("Genes evaluados:", len(expression_df))
print("Total de DEGs:", len(degs_df))
print(
    "Genes inducidos:",
    (degs_df["log2FoldChange"] > 1).sum()
)
print(
    "Genes reprimidos:",
    (degs_df["log2FoldChange"] < -1).sum()
)

display(degs_df.head())

Genes evaluados: 5394
Total de DEGs: 1458
Genes inducidos: 830
Genes reprimidos: 628


,locus_key,baseMean,log2FoldChange,lfcSE,stat,pvalue,adj_pvalue,regulacion,gene,final_product,replicon
0,PSPTO2135,11757.942868,6.987669,0.215518,32.422625,1.317427e-230,6.543662e-227,inducido,NaN,non-ribosomal peptide synthetase,NC_004578.1
1,PSPTO0679,2302.769106,-6.513743,0.210856,-30.891927,1.533218e-209,3.807748e-206,reprimido,NaN,OmpW/AlkL family protein,NC_004578.1
2,PSPTO4709,9429.024551,5.679935,0.196752,28.868511,2.967941e-183,4.913921e-180,inducido,cmaA,L-allo-isoleucine:holo-[CmaA peptidyl-carrier ...,NC_004578.1
3,PSPTO2152,5014.806817,7.579949,0.267562,28.329646,1.491247e-176,1.851755e-173,inducido,NaN,TonB-dependent siderophore receptor,NC_004578.1
4,PSPTO1232,12073.882594,6.314950,0.226863,27.835915,1.595180e-170,1.584652e-167,inducido,NaN,mannose-1-phosphate guanylyltransferase/mannos...,NC_004578.1


# Celda 2: Genes con función desconocida

In [8]:
from functional_analysis import (
    identify_unknown_function_genes,
    rank_unknown_function_genes
)

# Identificar genes de función desconocida o poco caracterizada
unknown_genes_df = identify_unknown_function_genes(degs_df)

# Ordenar los candidatos por magnitud del cambio de expresión
ranked_unknown_df = rank_unknown_function_genes(
    unknown_genes_df
)

# Seleccionar las columnas de interés
candidate_columns = [
    "locus_key",
    "final_product",
    "log2FoldChange",
    "adj_pvalue",
    "baseMean",
    "regulacion"
]

# Mostrar un resumen de los genes identificados
print("Total de DEGs de función desconocida:", len(unknown_genes_df))

print("\nDistribución por regulación:")
print(unknown_genes_df["regulacion"].value_counts())

print("\nAnotaciones más frecuentes:")
display(unknown_genes_df["final_product"].value_counts().head(10))

# Mostrar los 20 candidatos con mayor magnitud de cambio
print("\nPrincipales candidatos:")
display(ranked_unknown_df[candidate_columns].head(20))

# Guardar los resultados en un archivo CSV
unknown_genes_file = TABLES_DIR / "degs_funcion_desconocida.csv"

ranked_unknown_df[candidate_columns].to_csv(
    unknown_genes_file,
    index=False
)

print("\nArchivo guardado:", unknown_genes_file)

Total de DEGs de función desconocida: 182

Distribución por regulación:
regulacion
reprimido    128
inducido      54
Name: count, dtype: int64

Anotaciones más frecuentes:


final_product
hypothetical protein                 122
DUF1652 domain-containing protein      3
DUF2790 domain-containing protein      2
DUF2753 family protein                 1
DUF2726 domain-containing protein      1
DUF3828 domain-containing protein      1
DUF2474 domain-containing protein      1
DUF4868 domain-containing protein      1
DUF3325 domain-containing protein      1
DUF3757 domain-containing protein      1
Name: count, dtype: int64


Principales candidatos:


,locus_key,final_product,log2FoldChange,adj_pvalue,baseMean,regulacion
534,PSPTO2142,DUF6162 family protein,8.635782,7.858281e-08,55.076821,inducido
731,PSPTO1210,DUF3649 domain-containing protein,7.089940,5.116246e-06,50.595597,inducido
217,PSPTO1385,hypothetical protein,6.324478,2.924614e-16,291.296081,inducido
77,PSPTO4460,DUF2753 family protein,6.053724,8.628534e-40,557.809038,inducido
617,PSPTO3289,hypothetical protein,5.984415,6.467033e-07,46.584516,inducido
132,PSPTO1372,hypothetical protein,5.400927,2.046645e-24,175.917735,inducido
201,PSPTO5631,hypothetical protein,-5.221476,3.865635e-17,1100.232077,reprimido
1579,PSPTO0289,hypothetical protein,5.162102,1.387854e-02,5.328433,inducido
1461,PSPTO4458,hypothetical protein,4.660015,7.864504e-03,9.565497,inducido
587,PSPTO2143,hypothetical protein,4.511355,3.481579e-07,33.287328,inducido



Archivo guardado: ..\results\tables\degs_funcion_desconocida.csv


# Celda 3: Clasificación del T3SS

In [9]:
from functional_analysis import (
    identify_t3ss_genes,
    classify_t3ss_genes,
    summarize_t3ss_genes
)

# Identificar los candidatos relacionados con T3SS
t3ss_candidates_df = identify_t3ss_genes(degs_df)

# Clasificar los candidatos según su función
t3ss_classified_df = classify_t3ss_genes(
    t3ss_candidates_df
)

# Identificar los genes que requieren revisión manual
t3ss_review_df = t3ss_classified_df.loc[
    t3ss_classified_df["t3ss_category"] == "Revisar manualmente"
].copy()

# Separar los genes con categoría funcional asignada
t3ss_preliminary_df = t3ss_classified_df.loc[
    t3ss_classified_df["t3ss_category"] != "Revisar manualmente"
].copy()

# Obtener un resumen preliminar
t3ss_summary_df = summarize_t3ss_genes(
    t3ss_preliminary_df
)

# Mostrar los resultados
print("Candidatos iniciales:", len(t3ss_candidates_df))
print("Genes clasificados:", len(t3ss_preliminary_df))
print("Genes pendientes de revisión:", len(t3ss_review_df))

print("\nResumen funcional preliminar:")
display(t3ss_summary_df)

print("\nGenes pendientes de revisión:")
display(
    t3ss_review_df[
        [
            "locus_key",
            "gene",
            "final_product",
            "log2FoldChange",
            "regulacion"
        ]
    ]
)
# Conservar los genes clasificados del T3SS
t3ss_final_df = t3ss_preliminary_df.copy()

# Obtener el resumen funcional definitivo
t3ss_summary_df = summarize_t3ss_genes(t3ss_final_df)

# Seleccionar las columnas relevantes
t3ss_columns = [
    "locus_key",
    "gene",
    "final_product",
    "log2FoldChange",
    "adj_pvalue",
    "regulacion",
    "t3ss_category"
]

# Guardar la clasificación funcional
t3ss_final_file = TABLES_DIR / "degs_t3ss_clasificados.csv"

t3ss_final_df[t3ss_columns].to_csv(
    t3ss_final_file,
    index=False
)

# Guardar el resumen por categoría
t3ss_summary_file = TABLES_DIR / "resumen_t3ss.csv"

t3ss_summary_df.to_csv(t3ss_summary_file)

# Verificar los resultados
print("\nTotal de genes T3SS:", len(t3ss_final_df))
print("Genes inducidos:", (t3ss_final_df["regulacion"] == "inducido").sum())
print("Genes reprimidos:", (t3ss_final_df["regulacion"] == "reprimido").sum())

print("\nArchivos guardados:")
print(t3ss_final_file)
print(t3ss_summary_file)

ImportError: cannot import name 'identify_t3ss_genes' from 'functional_analysis' (C:\Users\norit\OneDrive\Documentos\LCG\Tercer_semestre\Biopy\Plant_pathogen_interaction\src\functional_analysis.py)

# Celda 4: Resumen funcional del T3SS

In [ ]:
from visualization import plot_t3ss_categories

# Crear la figura
fig, ax = plt.subplots(figsize=(11, 5))

# Representar las categorías funcionales del T3SS
plot_t3ss_categories(
    t3ss_summary_df,
    ax=ax
)

# Ajustar la distribución de los elementos
plt.tight_layout()

# Guardar la figura
figure_file = FIGURES_DIR / "t3ss_functional_categories.png"

fig.savefig(
    figure_file,
    dpi=300,
    bbox_inches="tight"
)

# Mostrar la gráfica
plt.show()

print("Figura guardada:", figure_file)